In [ ]:
"""
Bước 1: Data Loader
Dự án: Phân tích và dự đoán tỉ lệ nghỉ việc (IBM HR Attrition Enhanced)
bằng mô hình cây quyết định.

Chức năng:
    - Nạp file CSV vào DataFrame
    - Kiểm tra kích thước, kiểu dữ liệu, giá trị thiếu, dòng trùng lặp
    - Kiểm tra biến mục tiêu Attrition (phân phối, mức độ mất cân bằng)

Lưu ý: bước này CHỈ đọc và kiểm tra, không sửa đổi dữ liệu.
Việc làm sạch thuộc về bước Data Cleaner.
"""

from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt

# ====== CẤU HÌNH ======
DATA_PATH = "IBM_HR_Attrition_Enhanced.csv"  # đường dẫn local (dùng với load_data)
DRIVE_DATA_PATH = (
    "/content/drive/My Drive/Colab Notebooks/Project/IBM_HR_Attrition_Enhanced.csv"
)  # đường dẫn trên Google Drive (dùng với load_data_from_drive)
TARGET_COL = "Attrition"


# ====== 1. NẠP DỮ LIỆU ======
def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    """Đọc file CSV vào DataFrame, thử nhiều encoding phổ biến."""
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path.resolve()}")

    last_error = None
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"[OK] Đã nạp '{file_path.name}' (encoding={encoding})")
            return df
        except UnicodeDecodeError as e:
            last_error = e
    raise last_error


def load_data_from_drive(path: str = DRIVE_DATA_PATH) -> pd.DataFrame:
    """Mount Google Drive trong Colab rồi đọc CSV từ thư mục dự án.

    Đọc TOÀN BỘ các cột (không dùng usecols) để các bước sau có đủ dữ liệu.
    """
    try:
        from google.colab import drive
    except ImportError as e:
        raise RuntimeError(
            "Hàm này chỉ chạy được trong Google Colab. "
            "Nếu chạy trên máy local, hãy dùng load_data()."
        ) from e

    if not Path("/content/drive/My Drive").exists():
        drive.mount("/content/drive")
    else:
        print("[INFO] Google Drive đã được mount.")

    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file trên Drive: {file_path}\n"
            "Kiểm tra lại tên thư mục/tên file (phân biệt hoa thường)."
        )

    df = pd.read_csv(file_path, engine="python")
    print(f"[OK] Đã nạp '{file_path.name}' từ Google Drive")
    return df


# ====== 2. KIỂM TRA CẤU TRÚC ======
def check_shape(df: pd.DataFrame) -> None:
    rows, cols = df.shape
    print("\n" + "=" * 60)
    print("KÍCH THƯỚC DỮ LIỆU")
    print("=" * 60)
    print(f"Số dòng (nhân viên): {rows:,}")
    print(f"Số cột (thuộc tính): {cols}")
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Bộ nhớ sử dụng     : {mem_mb:.2f} MB")


def check_dtypes(df: pd.DataFrame) -> None:
    print("\n" + "=" * 60)
    print("KIỂU DỮ LIỆU")
    print("=" * 60)

    summary = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "n_unique": df.nunique(),
            "n_missing": df.isna().sum(),
            "pct_missing": (df.isna().mean() * 100).round(2),
        }
    )
    print(summary.to_string())

    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(exclude="number").columns.tolist()
    print(f"\nCột số      ({len(num_cols)}): {num_cols}")
    print(f"Cột phân loại ({len(cat_cols)}): {cat_cols}")


def check_quality(df: pd.DataFrame) -> None:
    """Kiểm tra nhanh: dòng trùng, giá trị thiếu, cột hằng số."""
    print("\n" + "=" * 60)
    print("CHẤT LƯỢNG DỮ LIỆU (chỉ thống kê, chưa xử lý)")
    print("=" * 60)

    n_dup = df.duplicated().sum()
    n_missing = int(df.isna().sum().sum())
    print(f"Dòng trùng lặp hoàn toàn: {n_dup}")
    print(f"Tổng số ô thiếu giá trị : {n_missing}")

    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print(f"Cột chỉ có 1 giá trị (không có tác dụng phân tách): {constant_cols or 'không có'}")

    # Cột gần như là mã định danh (mỗi dòng một giá trị khác nhau)
    id_like = [
        c for c in df.columns
        if df[c].nunique() == len(df) and c != TARGET_COL
    ]
    print(f"Cột giống mã định danh (nên loại ở bước cleaner): {id_like or 'không có'}")


# ====== 3. KIỂM TRA BIẾN MỤC TIÊU ======
def check_target(df: pd.DataFrame, target: str = TARGET_COL) -> None:
    print("\n" + "=" * 60)
    print(f"BIẾN MỤC TIÊU: {target}")
    print("=" * 60)

    if target not in df.columns:
        similar = [c for c in df.columns if target.lower() in c.lower()]
        raise KeyError(
            f"Không có cột '{target}'. Cột gần giống: {similar or 'không có'}. "
            f"Danh sách cột: {df.columns.tolist()}"
        )

    y = df[target]
    print(f"Kiểu dữ liệu     : {y.dtype}")
    print(f"Giá trị thiếu    : {y.isna().sum()}")
    print(f"Các giá trị duy nhất: {y.dropna().unique().tolist()}")

    counts = y.value_counts(dropna=False)
    pcts = y.value_counts(normalize=True, dropna=False) * 100
    dist = pd.DataFrame({"so_luong": counts, "ty_le_%": pcts.round(2)})
    print("\nPhân phối lớp:")
    print(dist.to_string())

    # Mức độ mất cân bằng
    vc = y.value_counts()
    if len(vc) >= 2:
        ratio = vc.max() / vc.min()
        print(f"\nTỉ lệ lớp đa số / lớp thiểu số = {ratio:.2f} : 1")
        if ratio >= 3:
            print("=> Dữ liệu MẤT CÂN BẰNG rõ rệt: cần bước Imbalance Handler, "
                  "và nên dùng F1/Recall/PR-AUC thay vì chỉ Accuracy.")
        elif ratio >= 1.5:
            print("=> Dữ liệu mất cân bằng nhẹ: nên cân nhắc class_weight hoặc stratify.")
        else:
            print("=> Dữ liệu tương đối cân bằng.")


# ====== 4. HÀM TỔNG HỢP ======
def run_data_loader(path: str = DRIVE_DATA_PATH, target: str = TARGET_COL) -> pd.DataFrame:
    """Chạy toàn bộ bước 1 và trả về DataFrame gốc (chưa chỉnh sửa)."""
    df = load_data_from_drive(path)

    print("\n5 dòng đầu:")
    print(df.head().to_string())

    check_shape(df)
    check_dtypes(df)
    check_quality(df)
    check_target(df, target)

    print("\n[HOÀN TẤT] Bước 1 - Data Loader.")
    return df


if __name__ == "__main__":
    df = run_data_loader()

In [ ]:
"""
Bước 2: Khám phá & làm sạch dữ liệu (Data Cleaner)
Dự án: Phân tích và dự đoán tỉ lệ nghỉ việc (IBM HR Attrition Enhanced)

Chức năng:
    1. Thống kê giá trị thiếu
    2. Phát hiện và loại bỏ bản ghi trùng lặp
    3. Phát hiện giá trị ngoại lai (IQR) - chỉ thống kê
    4. Phân phối biến mục tiêu Attrition
    5. Loại bỏ thuộc tính không mang thông tin dự báo:
       EmployeeCount, StandardHours, Over18, EmployeeNumber
"""

import pandas as pd
import matplotlib.pyplot as plt

TARGET_COL = "Attrition"
DROP_COLS = ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"]


# ====== 1. GIÁ TRỊ THIẾU ======
def report_missing(df: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "=" * 60)
    print("1. THỐNG KÊ GIÁ TRỊ THIẾU")
    print("=" * 60)

    missing = pd.DataFrame(
        {
            "n_missing": df.isna().sum(),
            "pct_missing": (df.isna().mean() * 100).round(2),
        }
    )
    missing = missing[missing["n_missing"] > 0].sort_values("n_missing", ascending=False)

    if missing.empty:
        print("Không có giá trị thiếu trong dữ liệu.")
    else:
        print(missing.to_string())
        print(f"\nTổng số ô thiếu: {int(missing['n_missing'].sum())}")
        print("=> Cần chọn chiến lược xử lý (điền trung vị/mode hoặc loại dòng).")
    return missing


# ====== 2. BẢN GHI TRÙNG LẶP ======
def handle_duplicates(df: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """Xử lý dòng trùng hoàn toàn (bao gồm cả cột định danh).

    Lưu ý: kiểm tra TRƯỚC khi bỏ EmployeeNumber, vì nếu bỏ mã nhân viên trước
    thì hai nhân viên khác nhau nhưng giống hệt các thuộc tính sẽ bị coi là trùng.
    """
    print("\n" + "=" * 60)
    print("2. BẢN GHI TRÙNG LẶP")
    print("=" * 60)

    n_dup = int(df.duplicated().sum())
    print(f"Số dòng trùng lặp hoàn toàn: {n_dup}")

    # Thông tin tham khảo: trùng khi bỏ các cột định danh/hằng số
    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    n_dup_feat = int(df.duplicated(subset=feature_cols).sum())
    print(f"Số dòng trùng nếu bỏ các cột {DROP_COLS}: {n_dup_feat} "
          "(chỉ để tham khảo, không loại bỏ)")

    if drop and n_dup > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"Đã loại {n_dup} dòng trùng. Kích thước mới: {df.shape}")
    return df


# ====== 3. GIÁ TRỊ NGOẠI LAI ======
def report_outliers(
    df: pd.DataFrame,
    target: str = TARGET_COL,
    k: float = 1.5,
    min_unique: int = 10,
) -> pd.DataFrame:
    """Phát hiện ngoại lai theo IQR: ngoài [Q1 - k*IQR, Q3 + k*IQR].

    Chỉ xét cột số liên tục (số giá trị duy nhất >= min_unique) để tránh
    các cột thang điểm/nhị phân. Không xóa dữ liệu: cây quyết định chia theo
    ngưỡng nên ít nhạy cảm với ngoại lai, và nhiều giá trị lớn (vd. lương,
    số năm kinh nghiệm) là thật chứ không phải lỗi.
    """
    print("\n" + "=" * 60)
    print("3. GIÁ TRỊ NGOẠI LAI (phương pháp IQR, chỉ thống kê)")
    print("=" * 60)

    num_cols = [
        c for c in df.select_dtypes(include="number").columns
        if c != target and c not in DROP_COLS and df[c].nunique() >= min_unique
    ]

    rows = []
    for c in num_cols:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - k * iqr, q3 + k * iqr
        n_out = int(((df[c] < low) | (df[c] > high)).sum())
        rows.append(
            {
                "column": c,
                "lower": round(low, 2),
                "upper": round(high, 2),
                "min": df[c].min(),
                "max": df[c].max(),
                "n_outliers": n_out,
                "pct_outliers": round(n_out / len(df) * 100, 2),
            }
        )

    out = pd.DataFrame(rows).sort_values("n_outliers", ascending=False)
    if out.empty:
        print("Không có cột số liên tục để kiểm tra.")
    else:
        print(out.to_string(index=False))
        n_cols_out = int((out["n_outliers"] > 0).sum())
        print(f"\n{n_cols_out}/{len(out)} cột có ngoại lai. Giữ nguyên dữ liệu.")
    return out


# ====== 4. PHÂN PHỐI BIẾN MỤC TIÊU ======
def report_target(df: pd.DataFrame, target: str = TARGET_COL, plot: bool = True) -> None:
    print("\n" + "=" * 60)
    print(f"4. PHÂN PHỐI BIẾN MỤC TIÊU: {target}")
    print("=" * 60)

    if target not in df.columns:
        raise KeyError(f"Không có cột '{target}' trong dữ liệu.")

    counts = df[target].value_counts()
    pcts = (df[target].value_counts(normalize=True) * 100).round(2)
    print(pd.DataFrame({"so_luong": counts, "ty_le_%": pcts}).to_string())

    if len(counts) >= 2:
        print(f"\nTỉ lệ lớp đa số / thiểu số = {counts.max() / counts.min():.2f} : 1")

    if plot:
        ax = counts.plot(kind="bar", color=["#4C78A8", "#E45756"][: len(counts)])
        ax.set_title(f"Phân phối {target}")
        ax.set_xlabel(target)
        ax.set_ylabel("Số nhân viên")
        for i, v in enumerate(counts):
            ax.text(i, v, f"{v} ({pcts.iloc[i]}%)", ha="center", va="bottom")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()


# ====== 5. LOẠI BỎ THUỘC TÍNH KHÔNG CÓ GIÁ TRỊ DỰ BÁO ======
def drop_useless_columns(df: pd.DataFrame, cols: list = None) -> pd.DataFrame:
    print("\n" + "=" * 60)
    print("5. LOẠI BỎ THUỘC TÍNH KHÔNG MANG THÔNG TIN DỰ BÁO")
    print("=" * 60)

    cols = cols or DROP_COLS
    present = [c for c in cols if c in df.columns]
    absent = [c for c in cols if c not in df.columns]

    for c in present:
        reason = "hằng số" if df[c].nunique(dropna=False) <= 1 else "mã định danh"
        print(f"- {c}: {df[c].nunique()} giá trị duy nhất ({reason})")
    if absent:
        print(f"[CẢNH BÁO] Không tìm thấy cột: {absent}")

    df = df.drop(columns=present)
    print(f"\nĐã loại {len(present)} cột. Kích thước mới: {df.shape}")
    return df


# ====== HÀM TỔNG HỢP ======
def run_data_cleaner(
    df: pd.DataFrame, target: str = TARGET_COL, plot: bool = True
) -> pd.DataFrame:
    """Chạy toàn bộ bước 2, trả về DataFrame đã làm sạch (bản sao, không sửa df gốc)."""
    df = df.copy()
    shape_before = df.shape

    report_missing(df)
    df = handle_duplicates(df)
    report_outliers(df, target)
    report_target(df, target, plot)
    df = drop_useless_columns(df)

    print("\n" + "=" * 60)
    print(f"[HOÀN TẤT] Bước 2 - Data Cleaner: {shape_before} -> {df.shape}")
    print("=" * 60)
    return df


if __name__ == "__main__":
    # from Data_loader import run_data_loader

    df_raw = run_data_loader()
    df_clean = run_data_cleaner(df_raw)

In [ ]:
"""
Bước 3: Mã hóa & biến đổi dữ liệu (Encoder)
Dự án: Phân tích và dự đoán tỉ lệ nghỉ việc (IBM HR Attrition Enhanced)

Chức năng:
    1. Chuyển biến mục tiêu Attrition sang nhị phân (Yes = 1, No = 0)
    2. Mã hóa One-Hot các thuộc tính phân loại danh định
    3. Giữ nguyên các thuộc tính số và thứ bậc (không chuẩn hóa vì cây
       quyết định chia theo ngưỡng, không phụ thuộc thang đo)
"""

import re

import pandas as pd

TARGET_COL = "Attrition"
TARGET_MAP = {"Yes": 1, "No": 0}

ONEHOT_COLS = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime",
    "AgeGroup",
    "TenureGroup",
    "MonthlyIncomeBracket",
]


# ====== 1. MÃ HÓA BIẾN MỤC TIÊU ======
def encode_target(df: pd.DataFrame, target: str = TARGET_COL) -> pd.DataFrame:
    print("\n" + "=" * 60)
    print(f"1. MÃ HÓA BIẾN MỤC TIÊU: {target} (Yes = 1, No = 0)")
    print("=" * 60)

    if target not in df.columns:
        raise KeyError(f"Không có cột '{target}' trong dữ liệu.")

    unexpected = set(df[target].dropna().unique()) - set(TARGET_MAP)
    if unexpected:
        raise ValueError(f"'{target}' có giá trị ngoài {list(TARGET_MAP)}: {unexpected}")
    if df[target].isna().any():
        raise ValueError(f"'{target}' có giá trị thiếu, cần xử lý trước khi mã hóa.")

    df = df.copy()
    df[target] = df[target].map(TARGET_MAP).astype(int)

    counts = df[target].value_counts().sort_index()
    for label, code in TARGET_MAP.items():
        n = counts.get(code, 0)
        print(f"{label:>3} -> {code}: {n} ({n / len(df) * 100:.2f}%)")
    return df


# ====== 2. ONE-HOT ======
def _clean_name(name: str) -> str:
    """Thay ký tự đặc biệt bằng '_' (vd. 'Established (6-10 yrs)' -> 'Established_6_10_yrs')."""
    return re.sub(r"[^0-9a-zA-Z_]+", "_", name).strip("_")


def onehot_encode(
    df: pd.DataFrame,
    cols: list = None,
    drop_first: bool = False,
) -> pd.DataFrame:
    """Mã hóa One-Hot các cột phân loại.

    drop_first=False (mặc định): giữ đủ k cột cho k giá trị. Với cây quyết
    định không cần tránh đa cộng tuyến, và giữ đủ cột giúp diễn giải
    (vd. 'OverTime_No' và 'OverTime_Yes' đều xuất hiện được trong luật chia).
    """
    print("\n" + "=" * 60)
    print("2. MÃ HÓA ONE-HOT")
    print("=" * 60)

    cols = cols or ONEHOT_COLS
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Không tìm thấy cột cần mã hóa: {missing}")

    n_cols_before = df.shape[1]
    for c in cols:
        print(f"- {c}: {df[c].nunique()} giá trị -> {df[c].nunique()} cột")

    df = pd.get_dummies(df, columns=cols, drop_first=drop_first, dtype=int)
    df.columns = [_clean_name(c) for c in df.columns]

    print(f"\nSố cột: {n_cols_before} -> {df.shape[1]}")
    return df


# ====== 3. KIỂM TRA KẾT QUẢ ======
def validate_encoded(df: pd.DataFrame, target: str = TARGET_COL) -> None:
    print("\n" + "=" * 60)
    print("3. KIỂM TRA SAU MÃ HÓA")
    print("=" * 60)

    non_numeric = df.select_dtypes(exclude="number").columns.tolist()
    n_missing = int(df.isna().sum().sum())
    dup_cols = df.columns[df.columns.duplicated()].tolist()

    print(f"Kích thước: {df.shape}")
    print(f"Cột không phải số còn sót lại: {non_numeric or 'không có'}")
    print(f"Tổng số ô thiếu              : {n_missing}")
    print(f"Tên cột bị trùng             : {dup_cols or 'không có'}")
    print(f"Biến mục tiêu '{target}' kiểu  : {df[target].dtype}, giá trị {sorted(df[target].unique())}")

    if non_numeric or n_missing or dup_cols:
        raise ValueError("Dữ liệu sau mã hóa chưa hợp lệ, xem các mục ở trên.")
    print("\n=> Dữ liệu sẵn sàng cho bước chia tập (Splitter).")


# ====== HÀM TỔNG HỢP ======
def run_encoder(
    df: pd.DataFrame,
    target: str = TARGET_COL,
    drop_first: bool = False,
) -> pd.DataFrame:
    """Chạy toàn bộ bước 3, trả về DataFrame đã mã hóa (không sửa df gốc)."""
    shape_before = df.shape

    df_enc = encode_target(df, target)
    df_enc = onehot_encode(df_enc, drop_first=drop_first)
    validate_encoded(df_enc, target)

    print("\n" + "=" * 60)
    print(f"[HOÀN TẤT] Bước 3 - Encoder: {shape_before} -> {df_enc.shape}")
    print("=" * 60)
    return df_enc


if __name__ == "__main__":
    # from data_loader import run_data_loader
    # from data_cleaner import run_data_cleaner

    df_raw = run_data_loader()
    df_clean = run_data_cleaner(df_raw, plot=False)
    df_encoded = run_encoder(df_clean)

In [ ]:
"""
Bước 4: Chia tập dữ liệu (Splitter)
Dự án: Phân tích và dự đoán tỉ lệ nghỉ việc (IBM HR Attrition Enhanced)

Chia phân tầng (stratified) theo nhãn Attrition:
    70% train | 15% validation | 15% test
với random_state cố định để kết quả tái lập được.
"""

import pandas as pd
from sklearn.model_selection import train_test_split

TARGET_COL = "Attrition"
RANDOM_STATE = 42
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15


def split_data(
    df: pd.DataFrame,
    target: str = TARGET_COL,
    train_size: float = TRAIN_SIZE,
    val_size: float = VAL_SIZE,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Chia df thành train/val/test phân tầng theo nhãn.

    Thực hiện 2 lần chia:
        (1) tách train khỏi phần còn lại (val + test)
        (2) chia đôi phần còn lại thành val và test theo tỉ lệ val : test
    Cả hai lần đều dùng stratify để giữ nguyên tỉ lệ lớp.
    """
    if target not in df.columns:
        raise KeyError(f"Không có cột '{target}' trong dữ liệu.")
    if abs(train_size + val_size + test_size - 1.0) > 1e-9:
        raise ValueError("train_size + val_size + test_size phải bằng 1.")

    X = df.drop(columns=[target])
    y = df[target]

    # (1) train vs (val + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=val_size + test_size,
        stratify=y,
        random_state=random_state,
    )

    # (2) val vs test, từ phần còn lại
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=test_size / (val_size + test_size),
        stratify=y_temp,
        random_state=random_state,
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


def report_split(df, X_train, X_val, X_test, y_train, y_val, y_test) -> None:
    print("\n" + "=" * 60)
    print("KẾT QUẢ CHIA TẬP DỮ LIỆU")
    print("=" * 60)

    total = len(df)
    rows = []
    for name, X, y in [
        ("Train", X_train, y_train),
        ("Validation", X_val, y_val),
        ("Test", X_test, y_test),
    ]:
        n_pos = int((y == 1).sum())
        rows.append(
            {
                "tap": name,
                "so_dong": len(X),
                "ty_le_tong_%": round(len(X) / total * 100, 2),
                "so_cot": X.shape[1],
                "attrition=1": n_pos,
                "attrition=0": len(y) - n_pos,
                "ty_le_attrition_%": round(n_pos / len(y) * 100, 2),
            }
        )
    print(pd.DataFrame(rows).to_string(index=False))
    print(f"\nToàn bộ dữ liệu: {total} dòng, "
          f"tỉ lệ Attrition = {df[TARGET_COL].mean() * 100:.2f}%")


def validate_split(df, X_train, X_val, X_test, y_train, y_val, y_test) -> None:
    """Kiểm tra: đủ dòng, không trùng chỉ số giữa các tập, X và y khớp nhau."""
    idx_tr, idx_va, idx_te = set(X_train.index), set(X_val.index), set(X_test.index)

    assert len(idx_tr) + len(idx_va) + len(idx_te) == len(df), "Tổng số dòng không khớp."
    assert not (idx_tr & idx_va or idx_tr & idx_te or idx_va & idx_te), \
        "Có dòng xuất hiện ở nhiều tập (rò rỉ dữ liệu)."
    assert (X_train.index == y_train.index).all(), "X_train và y_train lệch chỉ số."
    assert (X_val.index == y_val.index).all(), "X_val và y_val lệch chỉ số."
    assert (X_test.index == y_test.index).all(), "X_test và y_test lệch chỉ số."
    assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), \
        "Các tập có cột khác nhau."
    print("\n[OK] Kiểm tra hợp lệ: đủ dòng, không trùng lặp giữa các tập, X/y khớp nhau.")


# ====== HÀM TỔNG HỢP ======
def run_splitter(
    df: pd.DataFrame,
    target: str = TARGET_COL,
    random_state: int = RANDOM_STATE,
):
    """Chạy toàn bộ bước 4, trả về (X_train, X_val, X_test, y_train, y_val, y_test)."""
    print(f"Chia phân tầng {TRAIN_SIZE:.0%}/{VAL_SIZE:.0%}/{TEST_SIZE:.0%} "
          f"theo '{target}', random_state={random_state}")

    splits = split_data(df, target, random_state=random_state)
    report_split(df, *splits)
    validate_split(df, *splits)

    print("\n" + "=" * 60)
    print("[HOÀN TẤT] Bước 4 - Splitter")
    print("=" * 60)
    return splits


if __name__ == "__main__":
    # from data_loader import run_data_loader
    # from data_cleaner import run_data_cleaner
    # from encoder import run_encoder

    df_raw = run_data_loader()
    df_clean = run_data_cleaner(df_raw, plot=False)
    df_encoded = run_encoder(df_clean)
    X_train, X_val, X_test, y_train, y_val, y_test = run_splitter(df_encoded)

In [12]:
"""
Bước 5: Xử lý mất cân bằng lớp (Imbalance Handler)
Dự án: Phân tích và dự đoán tỉ lệ nghỉ việc (IBM HR Attrition Enhanced)

Phương án chính : trọng số lớp (class_weight='balanced')
Phương án so sánh: SMOTE (lấy mẫu tổng hợp lớp thiểu số)

Nguyên tắc: cả hai chỉ áp dụng trên tập HUẤN LUYỆN. Tập validation và test
giữ nguyên phân phối thực để các chỉ số đánh giá phản ánh đúng thực tế.
"""

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

RANDOM_STATE = 42


# ====== TIỆN ÍCH ======
def _class_dist(y: pd.Series) -> dict:
    vc = pd.Series(y).value_counts().sort_index()
    return {int(k): int(v) for k, v in vc.items()}


def _print_dist(name: str, y: pd.Series) -> None:
    d = _class_dist(y)
    total = sum(d.values())
    n0, n1 = d.get(0, 0), d.get(1, 0)
    print(f"{name:<28}: n={total:>5} | No(0)={n0:>5} | Yes(1)={n1:>4} "
          f"| tỉ lệ Yes={n1 / total * 100:5.2f}%")


# ====== 1. TRỌNG SỐ LỚP (PHƯƠNG ÁN CHÍNH) ======
def compute_balanced_weights(y_train: pd.Series) -> dict:
    """Tính trọng số 'balanced': w_c = n_samples / (n_classes * n_c).

    Chỉ để hiển thị/báo cáo. Khi huấn luyện chỉ cần truyền
    class_weight='balanced' cho DecisionTreeClassifier, kết quả tương đương.
    """
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    return {int(c): float(w) for c, w in zip(classes, weights)}


# ====== 2. SMOTE (PHƯƠNG ÁN SO SÁNH) ======
def apply_smote(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    variant: str = "smote",
    k_neighbors: int = 5,
    random_state: int = RANDOM_STATE,
):
    """Sinh mẫu tổng hợp cho lớp thiểu số, CHỈ trên tập huấn luyện.

    variant="smote"  : SMOTE chuẩn (nội suy mọi cột; các cột One-Hot 0/1 có
                       thể thành giá trị lẻ như 0.37).
    variant="smotenc": SMOTENC, coi các cột nhị phân 0/1 là biến phân loại nên
                       mẫu tổng hợp vẫn chỉ nhận 0 hoặc 1.
    """
    try:
        from imblearn.over_sampling import SMOTE, SMOTENC
    except ImportError as e:
        raise ImportError(
            "Cần cài imbalanced-learn: chạy '!pip install imbalanced-learn' "
            "trong một cell Colab rồi chạy lại."
        ) from e

    if variant == "smote":
        sampler = SMOTE(k_neighbors=k_neighbors, random_state=random_state)
    elif variant == "smotenc":
        is_binary = [set(X_train[c].unique()) <= {0, 1} for c in X_train.columns]
        sampler = SMOTENC(
            categorical_features=is_binary,
            k_neighbors=k_neighbors,
            random_state=random_state,
        )
    else:
        raise ValueError("variant phải là 'smote' hoặc 'smotenc'.")

    X_res, y_res = sampler.fit_resample(X_train, y_train)
    X_res = pd.DataFrame(X_res, columns=X_train.columns).reset_index(drop=True)
    y_res = pd.Series(y_res, name=y_train.name).reset_index(drop=True)
    return X_res, y_res


# ====== HÀM TỔNG HỢP ======
def run_imbalance_handler(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    smote_variant: str = "smote",
    random_state: int = RANDOM_STATE,
) -> dict:
    """Chuẩn bị các phương án huấn luyện để so sánh ở bước sau.

    Trả về dict, mỗi phương án gồm:
        X_train, y_train : dữ liệu huấn luyện của phương án đó
        class_weight     : giá trị truyền cho DecisionTreeClassifier
    Tập val/test không bị thay đổi, dùng chung cho mọi phương án.
    """
    print("\n" + "=" * 60)
    print("1. PHÂN PHỐI LỚP TRƯỚC KHI XỬ LÝ")
    print("=" * 60)
    _print_dist("Train", y_train)
    _print_dist("Validation (giữ nguyên)", y_val)
    _print_dist("Test (giữ nguyên)", y_test)

    # --- Phương án chính: class_weight ---
    print("\n" + "=" * 60)
    print("2. PHƯƠNG ÁN CHÍNH: class_weight='balanced'")
    print("=" * 60)
    weights = compute_balanced_weights(y_train)
    print(f"Trọng số lớp tính được: No(0) = {weights[0]:.3f}, Yes(1) = {weights[1]:.3f}")
    print(f"=> Sai sót ở lớp Yes bị phạt nặng gấp {weights[1] / weights[0]:.2f} lần lớp No.")
    print("Dữ liệu huấn luyện giữ nguyên, chỉ thay đổi hàm mục tiêu khi huấn luyện.")

    # --- Phương án so sánh: SMOTE ---
    print("\n" + "=" * 60)
    print(f"3. PHƯƠNG ÁN SO SÁNH: {smote_variant.upper()} (chỉ trên tập train)")
    print("=" * 60)
    X_smote, y_smote = apply_smote(
        X_train, y_train, variant=smote_variant, random_state=random_state
    )
    _print_dist("Train trước SMOTE", y_train)
    _print_dist("Train sau SMOTE", y_smote)
    n_synth = len(y_smote) - len(y_train)
    print(f"Số mẫu tổng hợp được thêm vào lớp Yes: {n_synth}")

    if smote_variant == "smote":
        onehot_like = [c for c in X_train.columns if set(X_train[c].unique()) <= {0, 1}]
        n_frac = sum(
            not np.isin(X_smote[c].iloc[len(X_train):].round(6), [0, 1]).all()
            for c in onehot_like
        )
        print(f"[LƯU Ý] {n_frac}/{len(onehot_like)} cột nhị phân có giá trị lẻ ở mẫu tổng hợp "
              "(đặc trưng của SMOTE chuẩn). Dùng smote_variant='smotenc' để tránh.")

    # --- Kiểm tra ---
    assert _class_dist(y_smote)[0] == _class_dist(y_smote)[1], "SMOTE chưa cân bằng 1:1."
    assert len(X_smote) == len(y_smote), "X và y sau SMOTE lệch độ dài."
    print("\n[OK] Tập validation và test không bị thay đổi ở cả hai phương án.")

    strategies = {
        "baseline": {  # tham chiếu: không xử lý gì
            "X_train": X_train, "y_train": y_train, "class_weight": None,
        },
        "class_weight": {  # phương án chính
            "X_train": X_train, "y_train": y_train, "class_weight": "balanced",
        },
        "smote": {  # phương án so sánh
            "X_train": X_smote, "y_train": y_smote, "class_weight": None,
        },
    }

    print("\n" + "=" * 60)
    print("[HOÀN TẤT] Bước 5 - Imbalance Handler")
    print("Các phương án: " + ", ".join(strategies))
    print("=" * 60)
    return strategies


if __name__ == "__main__":
    # from data_loader import run_data_loader
    # from data_cleaner import run_data_cleaner
    # from encoder import run_encoder
    # from splitter import run_splitter

    df_raw = run_data_loader()
    df_clean = run_data_cleaner(df_raw, plot=False)
    df_encoded = run_encoder(df_clean)
    X_train, X_val, X_test, y_train, y_val, y_test = run_splitter(df_encoded)
    strategies = run_imbalance_handler(X_train, y_train, X_val, y_val, X_test, y_test)

[INFO] Google Drive đã được mount.
[OK] Đã nạp 'IBM_HR_Attrition_Enhanced.csv' từ Google Drive

5 dòng đầu:
   Age Attrition     BusinessTravel  DailyRate              Department  DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  JobLevel                JobRole  JobSatisfaction MaritalStatus  MonthlyIncome  MonthlyRate  NumCompaniesWorked Over18 OverTime  PercentSalaryHike  PerformanceRating  RelationshipSatisfaction  StandardHours  StockOptionLevel  TotalWorkingYears  TrainingTimesLastYear  WorkLifeBalance  YearsAtCompany  YearsInCurrentRole  YearsSinceLastPromotion  YearsWithCurrManager AgeGroup             TenureGroup MonthlyIncomeBracket  OverallSatisfactionIndex
0   41       Yes      Travel_Rarely       1102                   Sales                 1          2  Life Sciences              1               1                        2  Female          94               3         2        Sales Executiv